# evaglass spot · Wan 2.2 B-roll (Kaggle GPU)

**Deneysel.** Wan 2.2 TI2V-5B'yi Kaggle T4 (16 GB) üzerinde diffusers ile çalıştırır. Kullanım: SH020 (karakter gözlüğü takar) ve SH040 (yakın plan konuşma) için yaşam tarzı B-roll. Referans görselden image-to-video ile üretim daha tutarlı sonuç verir; `REF_IMAGE` yolunu doldurun.

Notlar:
- T4 bf16'yı donanımda desteklemez; fp16 kullanılır, ilk denemede siyah kare çıkarsa `torch.float32` + `enable_sequential_cpu_offload()` ile yeniden deneyin (çok yavaş ama çalışır).
- Model ağırlıkları ~20 GB indirilir; Kaggle diski 70 GB'ın üstünde, sorun olmaz.
- 5 sn 720p klip T4'te 10-20 dk sürebilir. Önce 480p, 49 kare ile deneyin.
- Settings → Accelerator: GPU T4 x2, Internet: On.

In [ ]:
!pip -q install -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg
import torch, json, os
from diffusers import WanPipeline, WanImageToVideoPipeline, AutoencoderKLWan
from diffusers.utils import export_to_video, load_image
MODEL = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
DTYPE = torch.float16
REF_IMAGE = ""     # orn: /kaggle/input/evaglass-assets/character_ref.png  (bos: text-to-video)
WIDTH, HEIGHT, FRAMES, STEPS = 832, 480, 49, 30

In [ ]:
vae = AutoencoderKLWan.from_pretrained(MODEL, subfolder="vae", torch_dtype=torch.float32)
Pipe = WanImageToVideoPipeline if REF_IMAGE else WanPipeline
pipe = Pipe.from_pretrained(MODEL, vae=vae, torch_dtype=DTYPE)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
print("hazir")

In [ ]:
prompts = json.load(open("/kaggle/working/prompts.json")) if os.path.exists("/kaggle/working/prompts.json") else {
  "SH020": "Cinematic medium shot, a young Turkish woman in a dark modern living room picks up a pair of slim black smart glasses from a table and puts them on, soft tungsten key light from the left, cool blue rim light, shallow depth of field, slow crane reveal, 24fps, photorealistic, product stays sharp and unchanged",
  "SH040": "Cinematic close-up, the same woman wearing slim black smart glasses looks slightly off camera and speaks one short sentence, calm confident expression, static camera, 85mm lens, soft window light, photorealistic, subtle natural head motion"
}
NEG = "blurry, low quality, distorted face, extra fingers, watermark, text, logo, deformed glasses, flicker"
os.makedirs("/kaggle/working/broll", exist_ok=True)
for shot, prompt in prompts.items():
    kw = dict(prompt=prompt, negative_prompt=NEG, height=HEIGHT, width=WIDTH, num_frames=FRAMES, num_inference_steps=STEPS, guidance_scale=5.0)
    if REF_IMAGE: kw["image"] = load_image(REF_IMAGE).resize((WIDTH, HEIGHT))
    with torch.inference_mode():
        out = pipe(**kw).frames[0]
    path = f"/kaggle/working/broll/{shot}.mp4"
    export_to_video(out, path, fps=24)
    print("kaydedildi:", path)

Çıktılar `/kaggle/working/broll/` altında. Take beğenilmezse `generator=torch.Generator('cuda').manual_seed(N)` ekleyip seed değiştirerek 2-3 take alın. Kurguda 480p klipleri Resolve'un Super Scale özelliğiyle 1080p'ye büyütün.